# residual\_orrery — the twin-sphere residual-stream orrery (0.5B vs 1.5B)

A NOVEL mechanistic-interpretability visualization. Each Qwen2.5 decoder layer writes to the
**additive residual stream** twice (attention, then MLP). The MLP write is
`down_proj.weight @ a` — the *columns* of `down_proj` are **writer-direction** vectors in
$\mathbb{R}^{hidden}$. We fit one PCA(3) per model over the union of {residual trajectory,
down\_proj writer columns, target-token unembedding}, L2-normalize onto the unit sphere $S^2$,
then **animate a marker that hops the trajectory**, lighting up each layer's top-K writer-stars
as that layer fires. We render **0.5B vs 1.5B side by side** to compare how each model routes a
token through its writer constellation.

> Runs top-to-bottom on a **free Colab (CPU or T4)** — no GPU required. The 0.5B GIF takes ~30s;
> the side-by-side compare is dominated by 1.5B collection on CPU (first run ~10–12 min cold,
> seconds on a cache hit). `Runtime → Run all`.

**Version note (important):** the only version-sensitive call is in `animate.py`
(`_blacken_3d_axes` uses matplotlib's `ax.w_xaxis` / `w_yaxis` / `w_zaxis`, which were **removed in
matplotlib 3.8**). The install cell therefore pins `matplotlib<3.8`. Everything else (torch,
transformers, numpy, sklearn, imageio) works on modern Colab defaults. The package also avoids
`tokenizer.apply_chat_template` (it builds the Qwen2.5 ChatML prompt by hand), so old jinja2 is fine too.

## 0 · Hardware check (optional)
CPU is fine. A T4 only speeds up the 1.5B collection pass.

In [ ]:
!nvidia-smi || echo 'No GPU detected — that is OK, this notebook runs on CPU. (For a small speedup: Runtime → Change runtime type → T4 GPU.)'

## 1 · Clone the repo (branch `residual-orrery`)

In [ ]:
import os
REPO   = 'https://github.com/sinha-k-prat/small_fable.git'
BRANCH = 'residual-orrery'
if not os.path.isdir('small_fable'):
    !git clone -q -b $BRANCH $REPO
else:
    !cd small_fable && git fetch -q origin $BRANCH && git checkout -q $BRANCH && git pull -q
%cd small_fable
print('\nOn branch:')
!git rev-parse --abbrev-ref HEAD

## 2 · Install runtime deps + the package (`pip install -e .`)
We pin **`matplotlib<3.8`** (see the version note up top — `ax.w_xaxis` was removed in 3.8).
`scikit-learn` provides the PCA(3); `imageio` assembles the per-frame PNGs into a GIF (no ffmpeg).
`pip install -e .` reads the repo's `setup.py` and registers `import residual_orrery` +
`python -m residual_orrery`.

In [ ]:
# Runtime deps. torch/transformers/numpy are already on Colab; we only force the
# matplotlib cap and make sure sklearn + imageio + Pillow are present.
!pip install -q 'matplotlib<3.8' scikit-learn imageio Pillow 'transformers>=4.44' 'huggingface_hub'
# Editable install of the package itself (uses repo-root setup.py).
!pip install -q -e .
print('\nInstalled. Quick import check:')
import matplotlib, sklearn, imageio
print('matplotlib', matplotlib.__version__, '(must be < 3.8)')
print('sklearn   ', sklearn.__version__)
print('imageio   ', imageio.__version__)

## 2.5 · Torch-free smoke test (sanity, ~5s)
Exercises the `project` → `animate` pipeline on synthetic data — confirms PCA/sphere/slerp/GIF all
work on **this** Colab's matplotlib + imageio before we touch a real model. If this prints
`SMOKE OK`, the version-sensitive rendering path is good.

In [ ]:
!python -m residual_orrery --smoke

## 3 · Download the two Qwen2.5 models into the local HF cache
`residual_orrery.models.load_model` loads with `local_files_only=True` (it sets `HF_HUB_OFFLINE`
defensively). So we **pre-populate the cache** here with `snapshot_download`; afterwards the offline
load finds the files locally. These are public models — no token needed. (~1 GB + ~3 GB.)

In [ ]:
from huggingface_hub import snapshot_download
for repo in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']:
    print('downloading', repo, '...')
    snapshot_download(repo_id=repo,
                      allow_patterns=['*.json', '*.safetensors', '*.txt', 'tokenizer*', 'vocab*', 'merges*'])
print('\nBoth models cached locally.')

## 4 · Render the **single-panel 0.5B** orrery (fast, ~30s)
Good first artifact: wireframe $S^2$, the writer-star constellation, the white residual marker
hopping the trajectory, and per-layer top-K stars lighting up on the active layer. `--frames 55` is
auto-clamped up so every trajectory node is visited.

Example 0 is *"What is 17 plus 25? Reply with just the number."* (one greedy decode step → first
next-token, e.g. `'4'`).

In [ ]:
!python -m residual_orrery --models 0.5B --single --example 0 --frames 55 --topk 40 --dpi 110 --device cpu --out out

### Show the 0.5B GIF inline

In [ ]:
from IPython.display import Image, display
display(Image(filename='out/single_ex0_0.5B.gif'))

## 5 · Render the **0.5B vs 1.5B side-by-side** compare
Two 3D panels animating together — each model in its **own** PCA frame (H=896 vs H=1536 can't share
a basis), so this compares routing *shape*, not absolute aligned coordinates.

> **Timing:** on CPU the first run is dominated by 1.5B collection (per-layer `W_down @ a`
> reconstruction self-tests across 28 layers + a full-vocab argmax) — expect **~10–12 min cold**. A
> re-run hits the on-disk collection cache (`out/cache/*.npz`) and finishes in **~2 min** (still
> reloads both models). On a T4 it is faster.

In [ ]:
!python -m residual_orrery --models 0.5B 1.5B --example 0 --frames 55 --topk 40 --dpi 110 --device cpu --out out

### Show the side-by-side GIF inline

In [ ]:
from IPython.display import Image, display
display(Image(filename='out/compare_ex0_0.5B_vs_1.5B.gif'))

## 6 · (Optional) Try another example prompt
`--example` indexes the 5 built-in prompts (0..4):
`"17 plus 25"`, `"opposite of hot"`, `"capital of France is"`, `"is 7 prime"`, `"reverse 'cat'"`.
Or pass your own with `--prompt "..."`. Re-running reuses the cached collection, so only
projection + rendering re-run (seconds). Change the index below and run.

In [ ]:
EXAMPLE = 2  # 0..4  (2 = 'The capital of France is')
!python -m residual_orrery --models 0.5B 1.5B --example $EXAMPLE --frames 55 --topk 40 --dpi 110 --device cpu --out out
from IPython.display import Image, display
display(Image(filename=f'out/compare_ex{EXAMPLE}_0.5B_vs_1.5B.gif'))

## 6b · Render **all 5 examples** (side-by-side) and show each inline

One cell: renders the 0.5B-vs-1.5B compare for every prompt and displays each GIF.
On CPU this is ~1-3 min per example (~10 min total). Switch to a T4 (Runtime -> Change
runtime type -> T4 GPU) and set `DEVICE = 'cuda'` below for a big speedup.

In [ ]:
# Render ALL 5 examples (0.5B vs 1.5B) and show each inline.
from IPython.display import Image, display, Markdown

DEVICE = 'cpu'   # set to 'cuda' if you switched Runtime -> T4 GPU
PROMPTS = [
    "17 plus 25  (arithmetic)",
    "opposite of 'hot'  (antonym / lexical)",
    "capital of France  (factual recall)",
    "is 7 prime?  (binary reasoning)",
    "reverse 'cat'  (character-level)",
]
for ex in range(5):
    print(f'\n=== rendering example {ex}: {PROMPTS[ex]} ===')
    !python -m residual_orrery --models 0.5B 1.5B --example {ex} --frames 55 --topk 40 --dpi 110 --device {DEVICE} --out out
    display(Markdown(f'### Example {ex} — {PROMPTS[ex]}'))
    display(Image(filename=f'out/compare_ex{ex}_0.5B_vs_1.5B.gif'))
print('\nAll 5 rendered. GIFs are in out/.')


## 7 · (Optional) Download the GIFs
Saves the rendered artifacts to your machine from Colab.

In [ ]:
import glob
try:
    from google.colab import files
    for g in sorted(glob.glob('out/*.gif')):
        print('downloading', g); files.download(g)
except Exception as e:
    print('Not on Colab (or download skipped):', e)
    print('GIFs are on disk at:', sorted(glob.glob('out/*.gif')))